In [ ]:
import copy

import numpy as np
from matplotlib import pyplot as plt

%matplotlib inline
import taichi as ti

ti.init(
    arch=ti.cpu,
    default_fp=ti.f64,
    cpu_max_num_threads=1,
    offline_cache=False,
    debug=True,
)

from pespace.detector.antenna import InterferometerAntenna, FDResponseModelMarset2018
from pespace.detector.tdi import TDIChannelData, FDMichelsonConstantEqualArm
from pespace.detector.orbit import KaplerianHeliocentric
from tiwave.waveforms import IMRPhenomXAS

from bbhx.response.fastfdresponse import LISATDIResponse
from bbhx.waveformbuild import BBHWaveformFD
from  bbhx.waveforms.phenomhm import PhenomHMAmpPhase
import lal
import bilby

## Frequency-domain response

In [ ]:
f_ref = 1e-4
f_min = 1e-4
f_max = 0.1
t0 = 0.0

channels = ("A", "E", "T")
delta_time = 5
num_tsamples = 2 ** np.ceil(np.log2(4 * lal.DAYJUL_SI / delta_time))
duration = num_tsamples * delta_time
before_tc = 0.8 * duration
after_tc = 0.2 * duration
t_start = 0.0
tc = t_start + before_tc
print("sample num: ", num_tsamples)
print("duration: ", duration)
print("tc: ", tc)

params = dict(
    total_mass=3e6,
    mass_ratio=0.6,
    chi1_z=0.75,
    chi2_z=0.62,
    luminosity_distance=56000.0,
    inclination=0.4,
    reference_phase=1.3,
    ecliptic_longitude=1.375,
    ecliptic_latitude=-1.2108,
    polarization=2.659,
    coalescence_time=tc,
)
params = bilby.gw.conversion.generate_mass_parameters(params)
params

In [ ]:
tdi_data = TDIChannelData()
tdi_data.set_fd_data_from_zero(
    channels,
    duration,
    delta_time,
    start_time=t_start,
    minimum_frequency=f_min,
    maximum_frequency=f_max,
)
orbit_model = KaplerianHeliocentric(2.5e9, 0.0, 0.0)
response_model = FDResponseModelMarset2018()
tdi_combination = FDMichelsonConstantEqualArm(generation="2.0", orthogonal=True)

lisa = InterferometerAntenna(
    name="lisa",
    tdi_data=tdi_data,
    orbit_model=orbit_model,
    response_model=response_model,
    tdi_combination=tdi_combination,
)

In [ ]:
waveform_tiw = IMRPhenomXAS(tdi_data.frequency_samples, f_ref)
waveform_tiw.update_waveform(params)
f_peak_Hz = waveform_tiw.amplitude_coefficients[None].f_peak / waveform_tiw.source_parameters[None].M_sec  # fmt: skip
print("peak around (Hz): ", f_peak_Hz)

In [ ]:
waveform_tiw = IMRPhenomXAS(tdi_data.frequency_samples, f_peak_Hz)
waveform_tiw.update_waveform(params)

lisa.update_detector_response(
    waveform_tiw.waveform_container,
    params["ecliptic_longitude"],
    params["ecliptic_latitude"],
    params["polarization"],
    params["coalescence_time"],
)

In [ ]:
# abs
plt.figure()
plt.loglog(
    lisa.tdi_data.data_info.frequency_samples_array,
    np.abs(lisa.tdi_response_numpy["A"]),
    label="A",
)
plt.loglog(
    lisa.tdi_data.data_info.frequency_samples_array,
    np.abs(lisa.tdi_response_numpy["E"]),
    label="E",
)
plt.loglog(
    lisa.tdi_data.data_info.frequency_samples_array,
    np.abs(lisa.tdi_response_numpy["T"]),
    label="T",
)
plt.ylim(1e-28, 1e-16)
plt.xlim(f_min, f_max)
plt.legend()

# real part
plt.figure()
plt.semilogx(
    lisa.tdi_data.data_info.frequency_samples_array,
    lisa.tdi_response_numpy["A"].real,
    label="A",
)
plt.semilogx(
    lisa.tdi_data.data_info.frequency_samples_array,
    lisa.tdi_response_numpy["E"].real,
    label="E",
)
plt.semilogx(
    lisa.tdi_data.data_info.frequency_samples_array,
    lisa.tdi_response_numpy["T"].real,
    label="T",
)
plt.xlim(f_min, f_max)
plt.legend()

# imag part
plt.figure()
plt.semilogx(
    lisa.tdi_data.data_info.frequency_samples_array,
    lisa.tdi_response_numpy["A"].imag,
    label="A",
)
plt.semilogx(
    lisa.tdi_data.data_info.frequency_samples_array,
    lisa.tdi_response_numpy["E"].imag,
    label="E",
)
plt.semilogx(
    lisa.tdi_data.data_info.frequency_samples_array,
    lisa.tdi_response_numpy["T"].imag,
    label="T",
)
plt.xlim(f_min, f_max)
plt.legend()

## Comparing with BBHx

**difference in conventions from bbhx**
1. reference phase

2. Fourier transformation

To ensure the positive azimuthal modes (e.g. $h_{22}$) are concentrated in the positive frequency branch, bbhx uses the convention of Fourier transform with a different sign from the commonly used one (see appendix A in 1806.10734). 
While pespace adopt the conventional definition of FT. Thus there is a difference of global conjugation. It need to be noted that the waveform inputted also need the conjugation.

3. Unit vectors of the constellation (potential bug?? # TODO)

The unit vector of each link is computed by l.101 - 127 in d_EvaluateGslr of bbhx, e.g. n1 is obtained by orbits->get_normal_unit_vec(t, 12) which gives the vector between nodes 1 and 2. While in the computation of terms about sinc and exp (l.175 - 191), n1 is treated as the vector between nodes 2 and 3. Here we rigidly follow the computation in bbhx to verify whether other parts are all consistent, and leave this part in future checks.




In [ ]:
# make up a fake signal for comparision
amp = np.ones(tdi_data.data_info.frequency_series_length)
phi = np.zeros(tdi_data.data_info.frequency_series_length)
tf = np.ones(tdi_data.data_info.frequency_series_length)
h22 = amp * np.exp(1j * phi)

import taichi as ti
from tiwave.utils import ComplexNumber

harm_fac = ti.Struct.field({"plus": ComplexNumber, "cross": ComplexNumber}, shape=())


@ti.kernel
def compute_harmonic_factors():
    waveform_tiw.source_parameters[None].update_source_parameters(
        params["mass_1"],
        params["mass_2"],
        params["chi1_z"],
        params["chi2_z"],
        params["luminosity_distance"],
        params["inclination"],
        params["reference_phase"],
        f_ref,
    )
    waveform_tiw._set_harmonic_factors(harm_fac[None])  # depending the iota


compute_harmonic_factors()
harm_fac_np = harm_fac.to_numpy()
hp = harm_fac_np["plus"].view(np.complex128) * h22
hc = harm_fac_np["cross"].view(np.complex128) * h22
# hp = -1 * 0.125 * np.sqrt(5 / np.pi) * (1 + np.cos(params["inclination"]) ** 2) * h22
# hc = 1j * 0.125 * np.sqrt(5 / np.pi) * (2 * np.cos(params["inclination"])) * h22
hp = hp.view(np.float64).reshape(-1, 2)
hc = hc.view(np.float64).reshape(-1, 2)
wf_np = {
    "plus": hp,
    "cross": hc,
    "tf": tf,
}
wf_ti = ti.Struct.field(
    {
        "plus": ComplexNumber,
        "cross": ComplexNumber,
        "tf": ti.f64,
    },
    shape=tdi_data.frequency_samples.shape,
)
wf_ti.from_numpy(wf_np)

In [ ]:
import taichi.math as tm

from pespace.utils.utils import (
    get_polarization_tensor_ssb,
    get_gw_propagation_unit_vector,
    sinc,
    ComplexNumber,
)
from pespace.utils.constants import *


class ResponseModelBBHxConvention(FDResponseModelMarset2018):

    @ti.kernel
    def update_single_link_response(
        self,
        waveform: ti.template(),
        lam: ti.f64,
        beta: ti.f64,
        psi: ti.f64,
        tc: ti.f64,
    ):
        pol_tensor = get_polarization_tensor_ssb(lam, beta, psi)  # matrix: 3*3
        k = get_gw_propagation_unit_vector(lam, beta)  # vector: 3

        for i in self.detector.single_link_response:
            fi = self.detector.tdi_data.frequency_samples[i]
            cexp_tshift = tm.cexp(ComplexNumber([0.0, -2.0 * PI * fi * tc]))
            hp = tm.cmul(waveform[i].plus, cexp_tshift)
            hc = tm.cmul(waveform[i].cross, cexp_tshift)
            tf = waveform[i].tf + tc
            constellation_vectors = self.detector.orbit_model.get_constellation_vectors(tf)  # fmt: skip

            # n1: unit vector of 2 -> 3
            n1_h_n1 = (
                constellation_vectors.n1
                @ pol_tensor.plus
                @ constellation_vectors.n1
                * hp
                + constellation_vectors.n1
                @ pol_tensor.cross
                @ constellation_vectors.n1
                * hc
            )  # complex number
            # n2: unit vector of 3 -> 1
            n2_h_n2 = (
                constellation_vectors.n2
                @ pol_tensor.plus
                @ constellation_vectors.n2
                * hp
                + constellation_vectors.n2
                @ pol_tensor.cross
                @ constellation_vectors.n2
                * hc
            )  # complex number
            # n3: unit vector of 1 -> 2
            n3_h_n3 = (
                constellation_vectors.n3
                @ pol_tensor.plus
                @ constellation_vectors.n3
                * hp
                + constellation_vectors.n3
                @ pol_tensor.cross
                @ constellation_vectors.n3
                * hc
            )  # complex number

            k_n1 = k @ constellation_vectors.n1  # scalar
            k_n2 = k @ constellation_vectors.n2  # scalar
            k_n3 = k @ constellation_vectors.n3  # scalar

            k_x1_x2 = k @ (
                constellation_vectors.x1 + constellation_vectors.x2
            )  # scalar
            k_x2_x3 = k @ (
                constellation_vectors.x2 + constellation_vectors.x3
            )  # scalar
            k_x3_x1 = k @ (
                constellation_vectors.x3 + constellation_vectors.x1
            )  # scalar

            pi_f_L = PI * fi * self.detector.orbit_model.arm_length_sec  # scalar
            sinc32 = sinc(pi_f_L * (1.0 - k_n1))  # scalar
            sinc23 = sinc(pi_f_L * (1.0 + k_n1))  # scalar
            sinc13 = sinc(pi_f_L * (1.0 - k_n2))  # scalar
            sinc31 = sinc(pi_f_L * (1.0 + k_n2))  # scalar
            sinc21 = sinc(pi_f_L * (1.0 - k_n3))  # scalar
            sinc12 = sinc(pi_f_L * (1.0 + k_n3))  # scalar

            common_exp = -PI * fi * ComplexNumber([0.0, 1.0])  # ComplexNumber
            exp12 = tm.cexp(
                common_exp * (self.detector.orbit_model.arm_length_sec + k_x1_x2)
            )  # ComplexNumber
            exp23 = tm.cexp(
                common_exp * (self.detector.orbit_model.arm_length_sec + k_x2_x3)
            )  # ComplexNumber
            exp31 = tm.cexp(
                common_exp * (self.detector.orbit_model.arm_length_sec + k_x3_x1)
            )  # ComplexNumber

            prefactor = -pi_f_L * ComplexNumber([0.0, 1.0])  # ComplexNumber

            # self.detector.single_link_response[i].link12 = sinc12 * tm.cmul(
            #     tm.cmul(prefactor, n3_h_n3), exp12
            # )  # ComplexNumber
            # self.detector.single_link_response[i].link21 = sinc21 * tm.cmul(
            #     tm.cmul(prefactor, n3_h_n3), exp12
            # )  # ComplexNumber
            # self.detector.single_link_response[i].link23 = sinc23 * tm.cmul(
            #     tm.cmul(prefactor, n1_h_n1), exp23
            # )  # ComplexNumber
            # self.detector.single_link_response[i].link32 = sinc32 * tm.cmul(
            #     tm.cmul(prefactor, n1_h_n1), exp23
            # )  # ComplexNumber
            # self.detector.single_link_response[i].link31 = sinc31 * tm.cmul(
            #     tm.cmul(prefactor, n2_h_n2), exp31
            # )  # ComplexNumber
            # self.detector.single_link_response[i].link13 = sinc13 * tm.cmul(
            #     tm.cmul(prefactor, n2_h_n2), exp31
            # )  # ComplexNumber

            self.detector.single_link_response[i].link12 = sinc13 * tm.cmul(
                tm.cmul(prefactor, n2_h_n2), exp12
            )  # ComplexNumber
            self.detector.single_link_response[i].link21 = sinc31 * tm.cmul(
                tm.cmul(prefactor, n2_h_n2), exp12
            )  # ComplexNumber
            self.detector.single_link_response[i].link23 = sinc21 * tm.cmul(
                tm.cmul(prefactor, n3_h_n3), exp23
            )  # ComplexNumber
            self.detector.single_link_response[i].link32 = sinc12 * tm.cmul(
                tm.cmul(prefactor, n3_h_n3), exp23
            )  # ComplexNumber
            self.detector.single_link_response[i].link31 = sinc32 * tm.cmul(
                tm.cmul(prefactor, n1_h_n1), exp31
            )  # ComplexNumber
            self.detector.single_link_response[i].link13 = sinc23 * tm.cmul(
                tm.cmul(prefactor, n1_h_n1), exp31
            )  # ComplexNumber

In [ ]:
response_model = ResponseModelBBHxConvention()
tdi_combination = FDMichelsonConstantEqualArm(generation="2.0", orthogonal=True)
lisa_bbhx_convention = InterferometerAntenna(
    name="lisa_bbhx_convention",
    tdi_data=tdi_data,
    orbit_model=orbit_model,
    response_model=response_model,
    tdi_combination=tdi_combination,
)

lisa_bbhx_convention.update_detector_response(
    wf_ti,
    params["ecliptic_longitude"],
    params["ecliptic_latitude"],
    params["polarization"],
    0.0,
    )
chan1_pespace = lisa_bbhx_convention.tdi_response_numpy["A"]
chan2_pespace = lisa_bbhx_convention.tdi_response_numpy["E"]
chan3_pespace = lisa_bbhx_convention.tdi_response_numpy["T"]

In [ ]:
freqs = copy.deepcopy(tdi_data.data_info.frequency_samples_array)
response_kwargs = dict(TDItag="AET", tdi2=True)
response_bbhx = LISATDIResponse(**response_kwargs)

# due to the definition of FT, the phase differs by a conjugation
response_bbhx(
    freqs,
    params["inclination"],
    params["ecliptic_longitude"],
    params["ecliptic_latitude"],
    params["polarization"],
    # np.pi / 2-params['reference_phase'],
    np.pi / 2,
    len(freqs),
    modes=[(2, 2)],
    phase=-phi,
    tf=tf,
)
chan1_bbhx = response_bbhx.transferL1[0][0] * h22.conjugate()
chan2_bbhx = response_bbhx.transferL2[0][0] * h22.conjugate()
chan3_bbhx = response_bbhx.transferL3[0][0] * h22.conjugate()


In [ ]:
# abs
plt.figure()
plt.loglog(freqs, np.abs(chan1_pespace), label="chan1 (pespace)")
plt.loglog(freqs, np.abs(chan1_bbhx), label="chan1 (bbhx)")
plt.title("abs(chan1)")
plt.legend()

plt.figure()
plt.loglog(freqs, np.abs(chan2_pespace), label="chan2 (pespace)")
plt.loglog(freqs, np.abs(chan2_bbhx), label="chan2 (bbhx)")
plt.title("abs(chan2)")
plt.legend()

plt.figure()
plt.loglog(freqs, np.abs(chan3_pespace), label="chan3 (pespace)")
plt.loglog(freqs, np.abs(chan3_bbhx), label="chan3 (bbhx)")
plt.title("abs(chan3)")
plt.legend()

# real part
plt.figure()
plt.semilogx(freqs, chan1_pespace.real, label="chan1.real (pespace)")
plt.semilogx(freqs, chan1_bbhx.real, label="chan1.real (bbhx)")
plt.title("chan1 real")
plt.legend()

plt.figure()
plt.semilogx(freqs, chan2_pespace.real, label="chan2.real (pespace)")
plt.semilogx(freqs, chan2_bbhx.real, label="chan2.real (bbhx)")
plt.title("chan2 real")
plt.legend()

plt.figure()
plt.semilogx(freqs, chan3_pespace.real, label="chan3.real (pespace)")
plt.semilogx(freqs, chan3_bbhx.real, label="chan3.real (bbhx)")
plt.title("chan3 real")
plt.legend()

# imag part
plt.figure()
plt.semilogx(freqs, -chan1_pespace.imag, label="A.imag (pespace)")
plt.semilogx(freqs, chan1_bbhx.imag, label="A.imag (bbhx)")
plt.title("chan1 imag")
plt.legend()

plt.figure()
plt.semilogx(freqs, -chan2_pespace.imag, label="chan2.imag (pespace)")
plt.semilogx(freqs, chan2_bbhx.imag, label="chan2.imag (bbhx)")
plt.title("chan2 imag")
plt.legend()

plt.figure()
plt.semilogx(freqs, -chan3_pespace.imag, label="chan3.imag (pespace)")
plt.semilogx(freqs, chan3_bbhx.imag, label="chan3.imag (bbhx)")
plt.title("chan3 imag")
plt.legend()


In [ ]:
# wavefrom
wf_gen_bbhx = PhenomHMAmpPhase(run_phenomd=True)
wf_gen_bbhx(
    params['mass_1'],
    params['mass_2'],
    params['chi1_z'],
    params['chi2_z'],
    params['luminosity_distance']*1e6*lal.PC_SI,
    # params['reference_phase'],
    np.pi/2,
    f_peak_Hz,
    # params["coalescence_time"],
    0.0,
    len(freqs),
    freqs=freqs,
    modes=[(2,2)], 
    direct=True,
)


In [ ]:
params['reference_phase'] = 0.0
print(params)
waveform_tiw = IMRPhenomXAS(tdi_data.frequency_samples, f_peak_Hz, return_form='amplitude_phase')
waveform_tiw.update_waveform(params)

In [ ]:
plt.figure()
plt.loglog(freqs, wf_gen_bbhx.amp[0][0], label='amp (bbhx)')
plt.loglog(freqs, waveform_tiw.waveform_container_numpy["amplitude"], label='amp (tiwave)')
plt.legend()

plt.figure()
plt.semilogx(freqs, wf_gen_bbhx.phase[0][0], label='phase (bbhx)')
plt.semilogx(freqs, -waveform_tiw.waveform_container_numpy["phase"], label='phase (tiwave)')
plt.legend()

plt.figure()
plt.semilogx(freqs, wf_gen_bbhx.tf[0][0], label='tf (bbhx)')
plt.semilogx(freqs, waveform_tiw.waveform_container_numpy["tf"], label='tf (tiwave)')
plt.legend()

In [ ]:
# use a signal of GW from CBC
wave_gen = BBHWaveformFD(amp_phase_kwargs={'run_phenomd': True, }, 
                         response_kwargs={'TDItag':'AET', "tdi2": True, }, 
                         interp_kwargs={},
                         use_gpu=False)
tdi_responses_bbhx = wave_gen(
    params['mass_1'],
    params['mass_2'],
    params['chi1_z'],
    params['chi2_z'],
    params['luminosity_distance']*1e6*lal.PC_SI,
    # params['reference_phase'],
    -np.pi/2,
    f_peak_Hz,
    params['inclination'],
    params['ecliptic_longitude'],
    params['ecliptic_latitude'],
    params['polarization'],
    # params["coalescence_time"],
    0.0,
    freqs=freqs,
    modes=[(2,2)], 
    direct=True,
    t_obs_start=-1.0,
    t_obs_end=1.0,
    shift_t_limits=True,
)[0]


In [ ]:
params['reference_phase'] = 0.0
waveform_tiw = IMRPhenomXAS(tdi_data.frequency_samples, f_peak_Hz)
waveform_tiw.update_waveform(params)
lisa_bbhx_convention.update_detector_response(
    waveform_tiw.waveform_container,
    params["ecliptic_longitude"],
    params["ecliptic_latitude"],
    params["polarization"],
    # params['coalescence_time'],
    0.0,
    )

In [ ]:
# abs
plt.figure()
plt.loglog(freqs, np.abs(tdi_responses_bbhx[0]), label="chan1 (bbhx)")
plt.xlim(f_min, f_max)
plt.title("abs(chan1)")
plt.legend()

plt.figure()
plt.loglog(freqs, np.abs(tdi_responses_bbhx[1]), label="chan2 (bbhx)")
plt.xlim(f_min, f_max)
plt.title("abs(chan2)")
plt.legend()

plt.figure()
plt.loglog(freqs, np.abs(tdi_responses_bbhx[2]), label="chan3 (bbhx)")
plt.xlim(f_min, f_max)
plt.title("abs(chan3)")
plt.legend()

# real part
plt.figure()
plt.semilogx(freqs, tdi_responses_bbhx[0].real, label="chan1.real (bbhx)")
plt.xlim(f_min, f_max)
plt.title("chan1 real")
plt.legend()

plt.figure()
plt.semilogx(freqs, tdi_responses_bbhx[1].real, label="chan2.real (bbhx)")
plt.xlim(f_min, f_max)
plt.title("chan2 real")
plt.legend()

plt.figure()
plt.semilogx(freqs, tdi_responses_bbhx[2].real, label="chan3.real (bbhx)")
plt.xlim(f_min, f_max)
plt.title("chan3 real")
plt.legend()

# imag part
plt.figure()
plt.semilogx(freqs, tdi_responses_bbhx[0].imag, label="A.imag (bbhx)")
plt.xlim(f_min, f_max)
plt.title("chan1 imag")
plt.legend()

plt.figure()
plt.semilogx(freqs, tdi_responses_bbhx[1].imag, label="chan2.imag (bbhx)")
plt.xlim(f_min, f_max)
plt.title("chan2 imag")
plt.legend()

plt.figure()
plt.semilogx(freqs, tdi_responses_bbhx[2].imag, label="chan3.imag (bbhx)")
plt.xlim(f_min, f_max)
plt.title("chan3 imag")
plt.legend()


In [ ]:
# abs
plt.figure()
plt.loglog(freqs, np.abs(tdi_responses_bbhx[0]), label="chan1 (bbhx)")
plt.loglog(freqs, np.abs(lisa_bbhx_convention.tdi_response_numpy["A"]), linestyle='dashed', label="chan1 (pespace)")
plt.xlim(f_min, f_max)
plt.title("abs(chan1)")
plt.legend()

plt.figure()
plt.loglog(freqs, np.abs(tdi_responses_bbhx[1]), label="chan2 (bbhx)")
plt.loglog(freqs, np.abs(lisa_bbhx_convention.tdi_response_numpy["E"]), linestyle='dashed', label="chan2 (pespace)")
plt.xlim(f_min, f_max)
plt.title("abs(chan2)")
plt.legend()

plt.figure()
plt.loglog(freqs, np.abs(tdi_responses_bbhx[2]), label="chan3 (bbhx)")
plt.loglog(freqs, np.abs(lisa_bbhx_convention.tdi_response_numpy["T"]), linestyle='dashed', label="chan3 (pespace)")
plt.xlim(f_min, f_max)
plt.title("abs(chan3)")
plt.legend()

# real part
plt.figure()
plt.semilogx(freqs, tdi_responses_bbhx[0].real, label="chan1.real (bbhx)")
plt.semilogx(freqs, lisa_bbhx_convention.tdi_response_numpy["A"].real, linestyle='dashed', label="chan1.real (pespace)")
plt.xlim(f_min, f_max)
plt.title("chan1 real")
plt.legend()

plt.figure()
plt.semilogx(freqs, tdi_responses_bbhx[1].real, label="chan2.real (bbhx)")
plt.semilogx(freqs, lisa_bbhx_convention.tdi_response_numpy["E"].real, linestyle='dashed', label="chan2.real (pespace)")
plt.xlim(f_min, f_max)
plt.title("chan2 real")
plt.legend()

plt.figure()
plt.semilogx(freqs, tdi_responses_bbhx[2].real, label="chan3.real (bbhx)")
plt.semilogx(freqs, lisa_bbhx_convention.tdi_response_numpy["T"].real, linestyle='dashed', label="chan3.real (pespace)")
plt.xlim(f_min, f_max)
plt.title("chan3 real")
plt.legend()

# imag part
plt.figure()
plt.semilogx(freqs, tdi_responses_bbhx[0].imag, label="A.imag (bbhx)")
plt.semilogx(freqs, lisa_bbhx_convention.tdi_response_numpy["A"].imag, linestyle='dashed', label="chan1.imag (pespace)")
plt.xlim(f_min, f_max)
plt.title("chan1 imag")
plt.legend()

plt.figure()
plt.semilogx(freqs, tdi_responses_bbhx[1].imag, label="chan2.imag (bbhx)")
plt.semilogx(freqs, lisa_bbhx_convention.tdi_response_numpy["E"].imag, linestyle='dashed', label="chan1.imag (pespace)")
plt.xlim(f_min, f_max)
plt.title("chan2 imag")
plt.legend()

plt.figure()
plt.semilogx(freqs, tdi_responses_bbhx[2].imag, label="chan3.imag (bbhx)")
plt.semilogx(freqs, lisa_bbhx_convention.tdi_response_numpy["T"].imag, linestyle='dashed', label="chan1.imag (pespace)")
plt.xlim(f_min, f_max)
plt.title("chan3 imag")
plt.legend()


In [ ]:
plt.figure()
plt.loglog(freqs, np.abs(tdi_responses_bbhx[0].conjugate() - lisa_bbhx_convention.tdi_response_numpy["A"])/np.abs(tdi_responses_bbhx[0]), label="chan1 (abs diff)")
plt.xlim(f_min, f_max)
plt.legend()

plt.figure()
plt.loglog(freqs, np.abs(tdi_responses_bbhx[1].conjugate() - lisa_bbhx_convention.tdi_response_numpy["E"])/np.abs(tdi_responses_bbhx[1]), label="chan1 (abs diff)")
plt.xlim(f_min, f_max)
plt.legend()

plt.figure()
plt.loglog(freqs, np.abs(tdi_responses_bbhx[2].conjugate() - lisa_bbhx_convention.tdi_response_numpy["T"])/np.abs(tdi_responses_bbhx[2]), label="chan1 (abs diff)")
plt.xlim(f_min, f_max)
plt.legend()


## Time-domain response